In [72]:
!pip install polars

In [73]:
import polars as pl

In [74]:
df = pl.read_csv("/content/dirty_cafe_sales.csv")

In [75]:
print(df.columns)

['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']


In [76]:
print(df.dtypes)

[String, String, String, String, String, String, String, String]


In [77]:
df = df.with_columns([
    pl.when( pl.col("Item").is_in(["ERROR", "UNKNOWN"]) )
    .then(None)
    .otherwise(pl.col("Item"))
    .alias("Item")]
)

In [78]:
df = df.with_columns([
  pl.col("Quantity").cast(pl.Int64, strict=False),

  pl.col("Price Per Unit").cast(pl.Float64,strict=False),

  pl.col("Total Spent").cast(pl.Float64, strict=False),

  pl.col("Transaction Date").str.strptime(
      pl.Date,
      strict=False
  )
])

In [79]:
print(df.dtypes)

[String, String, Int64, Float64, Float64, String, String, Date]


In [80]:
print(df.shape)

(10000, 8)


In [81]:
print(df.null_count())

shape: (1, 8)
┌────────────────┬──────┬──────────┬───────────┬─────────────┬─────────┬──────────┬─────────────┐
│ Transaction ID ┆ Item ┆ Quantity ┆ Price Per ┆ Total Spent ┆ Payment ┆ Location ┆ Transaction │
│ ---            ┆ ---  ┆ ---      ┆ Unit      ┆ ---         ┆ Method  ┆ ---      ┆ Date        │
│ u32            ┆ u32  ┆ u32      ┆ ---       ┆ u32         ┆ ---     ┆ u32      ┆ ---         │
│                ┆      ┆          ┆ u32       ┆             ┆ u32     ┆          ┆ u32         │
╞════════════════╪══════╪══════════╪═══════════╪═════════════╪═════════╪══════════╪═════════════╡
│ 0              ┆ 969  ┆ 479      ┆ 533       ┆ 502         ┆ 2579    ┆ 3265     ┆ 460         │
└────────────────┴──────┴──────────┴───────────┴─────────────┴─────────┴──────────┴─────────────┘


In [82]:
df = df.with_columns(
  pl.when(pl.col("Quantity").is_null())
  .then(
      pl.col("Total Spent") / pl.col("Price Per Unit")
  )
  .otherwise(pl.col("Quantity"))
  .alias("Quantity")
)

In [83]:
df = df.with_columns(
    pl.when(pl.col("Price Per Unit").is_null())
    .then(
        pl.col("Total Spent") / pl.col("Quantity")
    )
    .otherwise(pl.col("Price Per Unit"))
    .alias("Price Per Unit")
)

In [84]:
df = df.with_columns(
    pl.when(pl.col("Total Spent").is_null())
    .then(
        pl.col("Quantity") * pl.col("Price Per Unit")
    )
    .otherwise(pl.col("Total Spent"))
    .alias("Total Spent")
)

In [85]:
df = df.with_columns(
    pl.when(
        pl.col("Payment Method")
        .is_in(["ERROR", "UNKNOWN"])
    )
    .then(None)
    .otherwise(pl.col("Payment Method"))
    .alias("Payment Method")
)

In [86]:
df = df.with_columns(
    pl.when(
        pl.col("Location")
        .is_in(["ERROR", "UNKNOWN"])
    )
    .then(None)
    .otherwise(pl.col("Location"))
    .alias("Location")
)

In [87]:
df = df.drop_nulls()

In [88]:
print(df.null_count())

shape: (1, 8)
┌────────────────┬──────┬──────────┬───────────┬─────────────┬─────────┬──────────┬─────────────┐
│ Transaction ID ┆ Item ┆ Quantity ┆ Price Per ┆ Total Spent ┆ Payment ┆ Location ┆ Transaction │
│ ---            ┆ ---  ┆ ---      ┆ Unit      ┆ ---         ┆ Method  ┆ ---      ┆ Date        │
│ u32            ┆ u32  ┆ u32      ┆ ---       ┆ u32         ┆ ---     ┆ u32      ┆ ---         │
│                ┆      ┆          ┆ u32       ┆             ┆ u32     ┆          ┆ u32         │
╞════════════════╪══════╪══════════╪═══════════╪═════════════╪═════════╪══════════╪═════════════╡
│ 0              ┆ 0    ┆ 0        ┆ 0         ┆ 0           ┆ 0       ┆ 0        ┆ 0           │
└────────────────┴──────┴──────────┴───────────┴─────────────┴─────────┴──────────┴─────────────┘


In [89]:
print(df.head)

<bound method DataFrame.head of shape: (3_556, 8)
┌──────────────┬──────────┬──────────┬──────────────┬───────┬─────────────┬──────────┬─────────────┐
│ Transaction  ┆ Item     ┆ Quantity ┆ Price Per    ┆ Total ┆ Payment     ┆ Location ┆ Transaction │
│ ID           ┆ ---      ┆ ---      ┆ Unit         ┆ Spent ┆ Method      ┆ ---      ┆ Date        │
│ ---          ┆ str      ┆ f64      ┆ ---          ┆ ---   ┆ ---         ┆ str      ┆ ---         │
│ str          ┆          ┆          ┆ f64          ┆ f64   ┆ str         ┆          ┆ date        │
╞══════════════╪══════════╪══════════╪══════════════╪═══════╪═════════════╪══════════╪═════════════╡
│ TXN_1961373  ┆ Coffee   ┆ 2.0      ┆ 2.0          ┆ 4.0   ┆ Credit Card ┆ Takeaway ┆ 2023-09-08  │
│ TXN_4977031  ┆ Cake     ┆ 4.0      ┆ 3.0          ┆ 12.0  ┆ Cash        ┆ In-store ┆ 2023-05-16  │
│ TXN_4271903  ┆ Cookie   ┆ 4.0      ┆ 1.0          ┆ 4.0   ┆ Credit Card ┆ In-store ┆ 2023-07-19  │
│ TXN_3160411  ┆ Coffee   ┆ 2.0      ┆ 2.

In [90]:
print(df.shape)

(3556, 8)


In [91]:
df.write_csv("cafe_sales_clean.csv")